# LLM B pilot

LLM B: `query + three result lists -> holistic preference` — sampled, sized sequentially to a
Wilson-interval target (decision 5, §7), a check on R1, never a full-pool leg. **Route names must be
blinded and list order swapped** (ADR 0001's amendment) — a judge told which list came from which
route re-encodes the query-shape heuristic the router must learn on its own.

**Primary judge: `google/gemini-3.1-pro-preview`** (top-ranked on RankJudge; also the frontier anchor
for the calibration spike — same model, one fewer moving part). **Optional second-family panel:
`moonshotai/kimi-k2.6`** — run both on the same rows to see whether they agree before paying for a
real panel at scale.

Direct OpenRouter REST calls (not litellm — same cost-calculator gap as LLM A). Tiny, real smoke
test — actual API calls, real dollars (expect well under a cent total). Proves the pipeline (blinding,
order-swap, position-to-route bookkeeping) and the real per-call price, not the list judge's agreement
rate — n here is nowhere near the ~200-300/stratum the plan sizes for.

In [ ]:
import os
import re
from random import Random

import pandas as pd
import requests
from dotenv import load_dotenv

from composition.pool_v3 import LabelledPool
from hybrid_search_rrf_dataset.labels import oracle_dir
from hybrid_search_rrf_dataset.router import DATA_DIR

load_dotenv()
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPEN_ROUTER_API_KEY")
assert OPENROUTER_KEY, "no OpenRouter key in .env"
COMPLETIONS_URL = "https://openrouter.ai/api/v1/chat/completions"

# (model, reasoning_effort) — gemini-3.1-pro-preview MANDATES reasoning on this
# endpoint (verified: effort='none' -> 400 "Reasoning is mandatory for this
# endpoint and cannot be disabled"); 'low' works and is the cheapest allowed
# setting. kimi-k2.6 does not reason by default, so 'none' is free to set.
JUDGES = {
    "gemini-3.1-pro": ("google/gemini-3.1-pro-preview", "low"),
    "kimi-k2.6": ("moonshotai/kimi-k2.6", "none"),
}
ROUTES = ["dense_only", "pure_rrf", "sparse_only"]
pd.set_option("display.width", 170)

In [ ]:
def complete(
    model: str, system: str, user: str, *, reasoning_effort: str = "none", max_tokens: int = 150,
) -> tuple[str, dict]:
    """`reasoning_effort` is per-model, not a global flag: `gemini-3.1-pro-preview`
    mandates reasoning on this endpoint (verified: `effort='none'` -> 400
    \"Reasoning is mandatory for this endpoint and cannot be disabled\"; `'low'`
    is the cheapest allowed setting and costs real reasoning tokens — measured
    85 reasoning + 4 visible tokens, $0.001164/call). `max_tokens=150` (not the
    LLM A pilot's 64) leaves room for that reasoning budget before the visible
    answer, on top of the empty-`content` failure LLM A's pilot hit at 64."""
    r = requests.post(
        COMPLETIONS_URL,
        headers={"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"},
        json={"model": model, "max_tokens": max_tokens, "usage": {"include": True},
              "reasoning": {"effort": reasoning_effort},
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": user}]},
        timeout=30,
    )
    r.raise_for_status()
    body = r.json()
    content = body["choices"][0]["message"]["content"]
    return (content or "").strip(), body.get("usage", {})

## 1. Draw — `routes_differ`-shaped rows, `native=False` only

`native=True` rows have zero oracle-ranking coverage (A5, measured 0% match vs 100% for
`native=False` on crumb-legal-qa) — the first version of this notebook drew without this filter and
got only 3/8 usable rows; fixed here.

In [ ]:
SEED = 0
N_ROWS = 8
LANE = "crumb-legal-qa"
TOP_N = 5  # short lists keep the smoke test cheap; the plan's real N is still open (question 7)

pool = LabelledPool()
classified = pool.classify(pool.labels())
lane_rows = classified[(classified["dataset"] == LANE) & ~classified["native"]]
candidates = lane_rows[lane_rows["kind"].isin(("decisive", "undecisive"))]  # differ, not tied/zero
drawn = candidates.sample(min(N_ROWS, len(candidates)), random_state=SEED)

oracle_rows = pd.read_parquet(
    oracle_dir(DATA_DIR / "route_labels", LANE) / "rows.parquet",
    columns=["query_id", "route_rankings"],
).astype({"query_id": str})
print(f"drew {len(drawn)} routes_differ-shaped rows")

## 2. Build the blinded, order-swapped prompt

Route -> letter is a fresh random permutation per row (`Random(seed).sample`), so which position
held which route cannot be memorized across rows — the point of the swap. Kept alongside the row for
unblinding after the reply, never shown to the model.

In [ ]:
corpus = pd.read_parquet(DATA_DIR / LANE / "corpus.parquet")

SNIPPET_CHARS = 200
"""Per-doc text shown to the judge. Deliberately far below the repo's 1200-char
`gold_text` convention and below R1's own `DOC_CHARS` — 15 snippets per call
(3 lists x TOP_N) means length multiplies fast. **This is a real limitation on
judgment quality, not only on cost:** 200 chars of a 1940-char-median legal
statute is roughly its heading, so the judge is largely comparing document
TITLES, not documents. It bounds what the measured agreement/tie rates below can
mean, and it is why the LLM B cost figure is a floor rather than an estimate
(open question 7 — list depth and snippet length are both unresolved)."""

corpus_text = {
    str(r.doc_id): (f"{r.title}\n{r.text}" if "title" in corpus.columns else str(r.text)).strip()[:SNIPPET_CHARS * 2]
    for r in corpus.itertuples()
}

INSTRUCTION = (
    "You compare three search result lists for the same query and judge which list is most useful "
    "overall — not just whether the top result is right, but the quality of the whole list. "
    "You are not told how any list was produced.\n\n"
    "Reply with EXACTLY ONE line: 'PREFER: A', 'PREFER: B', 'PREFER: C', or 'TIE'."
)


def build_prompt(row) -> tuple[str, dict[str, str]] | None:
    match = oracle_rows.loc[oracle_rows["query_id"] == str(row.query_id)]
    if match.empty:
        return None
    rankings = match.iloc[0]["route_rankings"]
    letters = Random(SEED + int(row.query_id)).sample(["A", "B", "C"], k=3)
    letter_to_route = dict(zip(letters, ROUTES))
    blocks = []
    for letter in ("A", "B", "C"):
        route = letter_to_route[letter]
        docs = list(rankings[route])[:TOP_N]
        snippets = [corpus_text.get(d, "")[:SNIPPET_CHARS] for d in docs]
        blocks.append(f"List {letter}:\n" + "\n---\n".join(s or "(empty)" for s in snippets))
    prompt = f"Query: {row.query}\n\n" + "\n\n".join(blocks)
    return prompt, letter_to_route

## 3. LIVE — spends real money, once per judge

In [ ]:
RUN_LIVE = False  # flip to True to actually spend

results = []
if RUN_LIVE:
    for role, (model, effort) in JUDGES.items():
        for row in drawn.itertuples():
            built = build_prompt(row)
            if built is None:
                continue
            prompt, letter_to_route = built
            text, usage = complete(model, INSTRUCTION, prompt, reasoning_effort=effort)
            results.append({"role": role, "model": model, "dataset": row.dataset,
                            "query_id": row.query_id, "kind": row.kind,
                            "llm_b_raw": text, "letter_to_route": letter_to_route,
                            "cost_usd": usage.get("cost", 0),
                            "prompt_tokens": usage.get("prompt_tokens"),
                            "completion_tokens": usage.get("completion_tokens")})
    results = pd.DataFrame(results)
    # an all-empty draw yields a column-less frame, and .groupby("role") then
    # raises KeyError instead of saying what actually went wrong
    if results.empty:
        raise RuntimeError(
            f"no calls made — {len(drawn)} rows drawn but every one failed "
            "build_prompt (no banked route_rankings? check the native=False filter)."
        )
    print(results.groupby("role")["cost_usd"].agg(["count", "sum", "mean"]))
else:
    print("RUN_LIVE is False — flip it above to actually call the API")

## 4. Unblind + compare, and check panel agreement (informal — n far too small for anything but pipeline proof)

In [ ]:
def unblind(raw: str, letter_to_route: dict[str, str]) -> str | None:
    m = re.match(r"PREFER:\s*([ABC])", raw, re.IGNORECASE)
    return letter_to_route[m.group(1).upper()] if m else None


if RUN_LIVE and len(results):
    results["llm_b_route"] = [
        unblind(r.llm_b_raw, r.letter_to_route) for r in results.itertuples()
    ]
    merged = results.merge(classified[["dataset", "query_id", "winner"]], on=["dataset", "query_id"], how="left")
    for role, group in merged.groupby("role"):
        ties = group["llm_b_route"].isna().sum()
        scored = group[group["llm_b_route"].notna()]
        line = f"{role:16s} ties/unparsed: {ties}/{len(group)}"
        if len(scored):
            agree = (scored["llm_b_route"] == scored["winner"]).mean()
            line += f"  agreement with spine on {len(scored)} non-tie rows: {agree:.0%}"
        print(line)

    # panel agreement: do the two judges pick the same route on the same rows?
    wide = merged.pivot_table(index=["dataset", "query_id"], columns="role", values="llm_b_route", aggfunc="first")
    if set(JUDGES) <= set(wide.columns):
        # dropna() removes any row where EITHER judge tied, so an empty overlap
        # is a realistic outcome (a judge that ties often shrinks it fast), not
        # an error — the per-judge rates above stay valid. Say so explicitly
        # rather than printing 'nan%', which reads like a failed computation.
        both = wide.dropna()
        first, second = list(JUDGES)
        if len(both):
            panel_agree = (both[first] == both[second]).mean()
            print(f"\npanel agreement between judges (n={len(both)}): {panel_agree:.0%}")
        else:
            tie_counts = {r: int(g["llm_b_route"].isna().sum()) for r, g in merged.groupby("role")}
            print(
                "\npanel agreement: NOT COMPUTABLE — no row has a non-tie verdict from "
                f"both judges (ties/unparsed per judge: {tie_counts}). "
                "The per-judge rates above are unaffected."
            )
    merged[["role", "dataset", "query_id", "kind", "llm_b_raw", "llm_b_route", "winner", "cost_usd"]]

## 5. Real cost, extrapolated to the plan's ~6,000-call sample scale

LLM B is a sample (decision 5), not a full-pool leg — extrapolate to the plan's own sizing
(~3,000 rows x 2 order-swaps = ~6,000 calls), not to 224,910.

In [ ]:
if RUN_LIVE and len(results):
    summary = results.groupby("role").agg(
        n=("cost_usd", "count"), per_call=("cost_usd", "mean"),
        mean_prompt_tok=("prompt_tokens", "mean"), mean_completion_tok=("completion_tokens", "mean"),
    )
    summary["extrapolated_6000"] = summary["per_call"] * 6_000
    print(summary.to_string())
    print("\nplan's hand estimate was: ~$43 (haiku)")